# whisper


In [ ]:
!pwd


### AMD GPU


In [ ]:
!sudo pacman -S --needed --noconfirm rocm-hip-sdk rocm-opencl-sdk ffmpeg


### conda env


In [ ]:
!conda create -n whisper python ipykernel -y


## rocm


In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.4
import torch
print(f"{torch.__version__=}")
print(f"{torch.cuda.is_available()=}")
print(f"{torch.cuda.get_device_name(0)=}" if torch.cuda.is_available() else "No GPU")


In [ ]:
%pip install openai-whisper


## env


## python 调用


In [8]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '10.3.0'
os.environ['http_proxy'] = 'socks5h://192.168.0.103:7897'
os.environ['https_proxy'] = 'socks5h://192.168.0.103:7897'

import whisper

download_root = "/data/.models/whisper"
os.makedirs(download_root, exist_ok=True)

model = whisper.load_model("base",device="cuda",download_root=download_root)
mp3="/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品/2022-03-02 20.20.24-视频-天启大烁哥-交易该怎么样复盘#期货 #交易员.mp3"
result = model.transcribe(mp3, language="zh")
print(result['text'])

from whisper.utils import get_writer
from pathlib import Path
# SRT 格式
output_dir = str(Path(mp3).parent)
srt_writer = get_writer("srt", output_dir)
srt_writer(result, mp3)
print(f"{mp3.replace('.mp3', '.srt')=}")


100%|███████████████████████████████████████| 139M/139M [00:17<00:00, 8.37MiB/s]


今天更各位分享一下我们该怎样去复盘复盘的核心非常简单就是你得带着自己的规则去复盘而不是去走视中寻找规律我从今就听过一个交易员的一个演讲他讲他自己怎么复盘的他打开了一段走视图指着这段走视图说你看这段走视图住了一个底那住底的过程是怎样发生的呢首先缩量创了一个心底然后再缩量又创了一个心底然后最后缩了一个更小的量又创了一个心底你看连续三次之后底部一般就形成了这个大家听得好像很开心的样子但是这个是真的吗这一个只是这一段走视的规律你去反看更多的总视你会发现有很多走视说的跟你一模一样但是持续不停的下跌这个并不是什么真正的规律你自从以觉得他是规律因为只在这一段走视中有效你只复盘了这一段走视可不是就形成了规律嘛对吧你要是一段走视的一个结果去定义全局的走视这个是非常不靠谱的这也是为什么交易这么拿的原因对吧因为你干不定就找不出那么一个规律但是很多人总是觉得自己可以找到得给他这样去做复盘很简单你要带着自己的交易规则去做你要去试错某一个方式比如说你找到一套方式就是海规交易法则吧突破二天最高点直接做多虽存两个ATR直接砍藏然后跌破时间的低点平藏你带着这个整套的规则去进行复盘你去复盘走视中我大量长期的用这种方法交易都发生了哪些事情在各种样的走视中都发生了哪些事情你会发现有很多的时候你都会亏都会资存两个ATR但是也有个别的时候你可能拿出来一波行情你通过大量的复盘大量的统计去进行这个过程之后你发现这个方法虽然是营营亏但是在最后可能就是曲线不停的向上的你会发现曲线不停向上并不代表着从来不亏可能是在营营亏亏之中累积出来的曲线的一个向上过程你这样不就通过复盘找到了一个方法到底靠不靠谱吗那你在这个过程中又有其他的想法了比如说你要试错什么WRH底然后是一个固定的营亏比比如说进场亏10%资存赚10%是指营用这种方式去不停的试各种各样的规则在各种各样的走势中会产生什么样的结果这个才是复盘的真正核心当然了因为现在即放一些的普及对吧有了量化测试可能会加速这一个过程但是总而言之复盘的核心是不变的就是你一定要先生成一个规则然后去试错这个规则而不是在混顿的走势中去慢无目的的寻找你就根本就寻找不出来什么绝对都规律的我们需要找到那种在大数定律之下在绝对多的走势的复盖之下在各种各样意外情况都包含的情况下依然能够通过大量的重复产生正向收益预期的交易逻辑和交易规则所以千万不要浪费时间去历史走势中寻找什么无敌的规律对吧去看着某一段走势

In [ ]:
import whisper
from whisper.utils import get_writer
from pathlib import Path
from tqdm import tqdm

def transcribe_to_srt(mp3_path, model):
    """单个音频转字幕"""
    mp3_path = Path(mp3_path)
    srt_path = mp3_path.with_suffix('.srt')
    
    if srt_path.exists():
        return srt_path
    
    result = model.transcribe(str(mp3_path), language="zh")
    
    output_dir = str(mp3_path.parent)
    srt_writer = get_writer("srt", output_dir)
    srt_writer(result, str(mp3_path))
    
    return srt_path

def batch_transcribe(folder_path, model_name="base"):
    """批量转字幕"""
    model = whisper.load_model(model_name, device="cpu")
    folder = Path(folder_path)
    mp3_files = list(folder.glob("*.mp3"))
    
    for mp3 in tqdm(mp3_files, desc="转字幕"):
        try:
            srt = transcribe_to_srt(mp3, model)
            print(f"{srt.name=}")
        except Exception as e:
            print(f"{mp3.name=} 失败: {e}")

# 使用
folder = "/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品"
batch_transcribe(folder)


## cli


In [1]:
!HSA_OVERRIDE_GFX_VERSION=10.3.0 whisper --help


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
usage: whisper [-h] [--model MODEL] [--model_dir MODEL_DIR] [--device DEVICE]
               [--output_dir OUTPUT_DIR]
               [--output_format {txt,vtt,srt,tsv,json,all}]
               [--verbose VERBOSE] [--task {transcribe,translate}]
               [--language {af,am,ar,as,az,ba,be,bg,bn,bo,br,bs,ca,cs,cy,da,de,el,en,es,et,eu,fa,fi,fo,fr,gl,gu,ha,haw,he,hi,hr,ht,hu,hy,id,is,it,ja,jw,ka,kk,km,kn,ko,la,lb,ln,lo,lt,lv,mg,mi,mk,ml,mn,mr,ms,mt,my,ne,nl,nn,no,oc,pa,pl,ps,pt,ro,ru,sa,sd,si,sk,sl,sn,so,sq,sr,su,sv,sw,ta,te,tg,th,tk,tl,tr,tt,uk,ur,uz,vi,yi,yo,yue,zh,Afrikaans,Albanian,Amharic,Arabic,Armenian,Assamese,Azerbaijani,Bashkir,Basque,Belarusian,Bengali,Bosnian,Breton,Bulgarian,Burmese,Cantonese,Castilian,Catalan,Chinese,Croatian,Czech,Danish,Dutch,English,Estonian,Faroese,Finnish,Flemish,French,Galician,Georgian,German,Greek,Gujarati,Haitian,Haitian Creole,Hausa,Hawaiian,Hebrew,Hindi,Hungarian,Icelandic,Indones

In [5]:
%%bash
all_proxy=socks5h://192.168.0.103:7897 HSA_OVERRIDE_GFX_VERSION=10.3.0 whisper \
"/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品/2022-03-02 20.20.24-视频-天启大烁哥-交易该怎么样复盘#期货 #交易员.mp3" \
 --language zh \
 --model turbo \
 --model_dir "/data/.models/whisper" \
 --device cuda \
 --output_format srt \
 --output_dir "/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品"


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


[00:00.060 --> 00:03.140] 今天跟各位分享一下我们该怎样去复盘
[00:03.140 --> 00:04.920] 复盘的核心非常简单
[00:04.920 --> 00:06.900] 就是你要带着自己的规则去复盘
[00:06.900 --> 00:08.380] 而不是去走势中寻找规律
[00:08.380 --> 00:11.940] 我曾经就听过一个交易员的一个演讲
[00:11.940 --> 00:13.700] 他讲他自己怎么复盘的
[00:13.700 --> 00:15.080] 他打开了一段走势图
[00:15.080 --> 00:16.140] 指的这段走势图说
[00:16.140 --> 00:17.820] 你看这段走势图柱了一个底
[00:17.820 --> 00:20.060] 那柱底的过程是怎样发生的呢
[00:20.060 --> 00:21.780] 首先缩量创了一个新低
[00:21.780 --> 00:23.320] 然后再缩量又创了一个新低
[00:23.320 --> 00:25.120] 然后最后缩了一个更小的量
[00:25.120 --> 00:26.100] 又创了一个新低
[00:26.100 --> 00:28.780] 你看连续三次之后底部一般就形成了
[00:28.780 --> 00:30.940] 这个大家听得好像很开心的样子
[00:30.940 --> 00:31.860] 但是这个是真的吗
[00:31.860 --> 00:33.960] 这个只是这一段走势的规律
[00:33.960 --> 00:35.220] 你去翻看更多的走势
[00:35.220 --> 00:37.260] 你会发现有很多走势说的跟你一模一样
[00:37.260 --> 00:38.840] 但是持续不停的下跌
[00:38.840 --> 00:40.980] 这个并不是什么真正的规律
[00:40.980 --> 00:42.640] 你之所以觉得它是规律
[00:42.640 --> 00:44.280] 因为只在这一段走势中有效
[00:44.280 --> 00:45.920] 你只复盘了这一段走势
[00:45.920 --> 00:47.500] 可不是就形成了规律吗